In [ ]:
# Install all necessary tools: AI Separation, Video Editing, and Gap Removal
!pip install -q demucs moviepy pydub diffq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import re
from pathlib import Path
from moviepy.editor import VideoFileClip
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

# 1. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --- DIRECTORY SETUP ---
BASE_PATH = "/content/drive/MyDrive/ProiectSpeech"
INPUT_DIR = f"{BASE_PATH}/ToBeConverted"
OUTPUT_DIR = f"{BASE_PATH}/AudioOutput"
STORAGE_DIR = f"{BASE_PATH}/Videos"
MODEL = "mdx"

for folder in [INPUT_DIR, OUTPUT_DIR, STORAGE_DIR]:
    os.makedirs(folder, exist_ok=True)

# 2. FIND AND SANITIZE
files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.mp4', '.mov', '.mkv', '.avi'))]

if not files:
    print(f"ToBeConverted is empty. Please place a video in {INPUT_DIR}")
else:
    original_filename = files[0]
    video_path = os.path.join(INPUT_DIR, original_filename)

    clean_base = re.sub(r'[^a-zA-Z0-9]', '_', Path(original_filename).stem)
    clean_base = re.sub(r'_+', '_', clean_base).strip('_')

    project_folder = os.path.join(OUTPUT_DIR, f"{clean_base}_Project")
    cuts_folder = os.path.join(project_folder, "AudioCuts")
    metadata_folder = os.path.join(project_folder, "Metadata")

    if os.path.exists(project_folder):
        print(f"Cleaning up existing project folder for '{clean_base}'...")
        shutil.rmtree(project_folder)

    os.makedirs(cuts_folder, exist_ok=True)
    os.makedirs(metadata_folder, exist_ok=True)

    print(f"Processing: {original_filename}")

    # 3. EXTRACT & SEPARATE
    video = VideoFileClip(video_path)
    temp_raw = "temp_proc.mp3"
    video.audio.write_audiofile(temp_raw, logger=None)

    print("AI is separating vocals...")
    !python3 -m demucs.separate --two-stems=vocals -n {MODEL} --shifts=2 --flac "{temp_raw}"

    # 4. DIALOGUE LOGIC
    vocals_flac = f"separated/{MODEL}/temp_proc/vocals.flac"

    if not os.path.exists(vocals_flac):
        print("Separation failed.")
    else:
        audio = AudioSegment.from_file(vocals_flac, format="flac")
        intervals = detect_nonsilent(audio, min_silence_len=1200, silence_thresh=-45)

        combined_audio = AudioSegment.empty()
        count = 0
        metadata_lines = [f"PROJECT: {clean_base}\n", "-"*40 + "\n"]

        for start_ms, end_ms in intervals:
            duration_ms = end_ms - start_ms
            if duration_ms < 400: continue

            count += 1
            pad = 500
            chunk = audio[max(0, start_ms - pad):min(len(audio), end_ms + pad)]

            # Clean internal naming
            chunk_name = f"Part{count:02d}.wav"
            chunk.export(os.path.join(cuts_folder, chunk_name), format="wav")

            s_m, s_s = divmod(start_ms // 1000, 60)
            e_m, e_s = divmod(end_ms // 1000, 60)
            pretty_time = f"{s_m:02d}:{s_s:02d} - {e_m:02d}:{e_s:02d}"
            metadata_lines.append(f"{chunk_name:<10} | {pretty_time}\n")
            combined_audio += chunk

        # 5. EXPORT
        if count > 0:
            combined_audio.export(os.path.join(project_folder, "Full_Vocal_Master.wav"), format="wav")
            # "Shuttle" file
            timeline_path = os.path.join(metadata_folder, f"{clean_base}_Timeline.txt")
            with open(timeline_path, "w") as f:
                f.writelines(metadata_lines)

            print(f"Done! Move '{clean_base}_Timeline.txt' to ToTranscribe to start Whisper.")

        # Cleanup
        os.remove(temp_raw)
        if os.path.exists("separated"): shutil.rmtree("separated")
        shutil.move(video_path, os.path.join(STORAGE_DIR, original_filename))

In [ ]:
!pip install openai-whisper

In [ ]:
import os
import shutil
import whisper
import re

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/ProiectSpeech"
TRIGGER_DIR = f"{BASE_PATH}/ToTranscribe"
OUTPUT_DIR = f"{BASE_PATH}/AudioOutput"

os.makedirs(TRIGGER_DIR, exist_ok=True)

print("Loading Whisper AI model...")
model = whisper.load_model("medium")

# Find the shuttle files
work_orders = [f for f in os.listdir(TRIGGER_DIR) if f.endswith("_Timeline.txt")]

if not work_orders:
    print(f"No files found in {TRIGGER_DIR}. Manually move a Timeline file here to begin.")
else:
    for order in work_orders:
        # Extract project identity
        project_name = order.replace("_Timeline.txt", "")
        project_folder = os.path.join(OUTPUT_DIR, f"{project_name}_Project")
        cuts_folder = os.path.join(project_folder, "AudioCuts")
        meta_folder = os.path.join(project_folder, "Metadata")
        whisper_folder = os.path.join(project_folder, "Whisper")

        if not os.path.exists(cuts_folder):
            print(f"Error: Could not find AudioCuts for {project_name}")
            continue

        print(f"\n--- Transcribing Project: {project_name} ---")
        os.makedirs(whisper_folder, exist_ok=True)

        audio_files = sorted([f for f in os.listdir(cuts_folder) if f.endswith(".wav")])

        # Identifiable Transcript naming
        transcript_path = os.path.join(whisper_folder, f"{project_name}_Transcript.txt")

        with open(transcript_path, "w") as f:
            f.write(f"WHISPER TRANSCRIPT: {project_name}\n")
            f.write("-" * 40 + "\n\n")

            for audio_file in audio_files:
                file_path = os.path.join(cuts_folder, audio_file)
                result = model.transcribe(file_path, fp16=False)
                text = result['text'].strip()

                part_label = audio_file.replace(".wav", "")
                line = f"[{part_label}]: {text}"
                print(line)
                f.write(line + "\n")

        # RETURN THE SHUTTLE: Move timeline file from "ToTranscribe" back to "Metadata"
        source_path = os.path.join(TRIGGER_DIR, order)
        destination_path = os.path.join(meta_folder, order)

        # Using shutil.move to handle the "return" trip
        shutil.move(source_path, destination_path)

        print(f"Done! {order} returned to Metadata. Transcript created.")

print("\nAll work orders processed.")